# 11 — TEST cuối: hệ thống đã chốt V4B và các mốc so sánh

Hệ thống cuối **V4B** (V4 = V3 + SpecAugment, giải mã beam 5 + guard) đã được chốt trên DEV trước khi giải mã TEST:
`ASR/reports/final_freeze_20261027/freeze.json` (adapter sha256, cách giải mã, số liệu DEV của mọi ứng viên).

Mỗi ô giải mã TEST và HARD (lát ViMedCSS khó, nằm trong TEST) **một lần**: nếu đã có kết quả, ô chỉ in đường dẫn, không chạy lại.
Sau khi đọc TEST không chỉnh model hay cách giải mã nữa.

**Trước khi chạy:** tắt kernel của notebook 08, 09, 10 (mở notebook → Restart/Shut down kernel, hoặc đóng hẳn notebook) và đóng Chrome.
Mỗi kernel cũ giữ ~0,6 GB; máy chỉ còn ~3 GB bộ nhớ commit trống, không đủ để nạp PyTorch + model (hai lần chạy nền đã chết vì vậy).
Thời gian ước tính ở DUTY 1,0: V4B ~3 giờ, B0V/V3/V4 ~1,5 giờ mỗi model, V3B ~3 giờ (tùy chọn, để đủ bảng 2×2).


In [1]:
from pathlib import Path
import sys, json, ctypes
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p/'asr/config.py').is_file()), None)
if PROJECT is None:
    PROJECT = next((p/'ASR' for p in [here, *here.parents] if (p/'ASR/asr/config.py').is_file()), None)
assert PROJECT is not None, 'Mở notebook từ trong workspace dự án'
PYTHON = PROJECT.parent/'.venv-asr/Scripts/python.exe'
sys.path.insert(0, str(PROJECT/'local'))
from notebook_process import run_cli as _run_cli
def run_cli(arguments):
    return _run_cli(PROJECT, PYTHON, arguments)
RUNNER = PROJECT/'scripts/local_run.py'
RUNS = PROJECT/'local/runs'
DUTY = '1.0'

freeze = json.loads((PROJECT/'reports/final_freeze_20261027/freeze.json').read_text(encoding='utf-8'))
print('Hệ thống cuối:', freeze['final_system'], '| chốt lúc', freeze['frozen_at'])
print('Adapter sha256:', freeze['adapter']['adapter_model.safetensors'])
print('Giải mã:', freeze['decode'])
print('WER DEV (%):', {k: round(100*v['wer'], 2) for k, v in freeze['dev_evidence'].items()})

class _MemStatus(ctypes.Structure):
    _fields_ = [('dwLength', ctypes.c_ulong), ('dwMemoryLoad', ctypes.c_ulong), ('ullTotalPhys', ctypes.c_ulonglong),
                ('ullAvailPhys', ctypes.c_ulonglong), ('ullTotalPageFile', ctypes.c_ulonglong),
                ('ullAvailPageFile', ctypes.c_ulonglong), ('ullTotalVirtual', ctypes.c_ulonglong),
                ('ullAvailVirtual', ctypes.c_ulonglong), ('ullAvailExtendedVirtual', ctypes.c_ulonglong)]

def commit_free_gb():
    s = _MemStatus(); s.dwLength = ctypes.sizeof(s)
    ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(s))
    return s.ullAvailPageFile / 1e9

free = commit_free_gb()
print(f'Bộ nhớ commit trống: {free:.1f} GB', '' if free >= 5 else '-> THẤP: tắt kernel notebook cũ và Chrome rồi chạy lại cell này')

def gpu_job_running():
    import psutil
    lock = PROJECT/'local/gpu.lock'
    if not lock.exists():
        return None
    pid = int(lock.read_text())
    return pid if psutil.pid_exists(pid) else None

def decode_once(exp, split, run_id, *extra):
    """Decode one split once; skip if its metrics already exist (TEST is not re-run)."""
    out = RUNS/exp/run_id/'eval'/split/'metrics.json'
    if out.is_file():
        print('Đã có, không chạy lại:', out); return
    if gpu_job_running():
        print('Đang có job GPU khác; đợi nó xong.'); return
    if commit_free_gb() < 4:
        print(f'Bộ nhớ commit chỉ còn {commit_free_gb():.1f} GB; tắt bớt ứng dụng/kernel rồi chạy lại cell.'); return
    run_cli([RUNNER, 'eval', exp, '--split', split, '--allow-test', '--run-id', run_id, '--duty-cycle', DUTY, *extra])

V3_FINAL = RUNS/'V3_phowhisper_small_lora_vsplit/local_v3/final'
V4_FINAL = RUNS/'V4_phowhisper_small_lora_vsplit_specaug/local_v4/final'


Hệ thống cuối: V4B_phowhisper_small_lora_vsplit_specaug_beam5 | chốt lúc 2026-10-27T01:07:57
Adapter sha256: 5667e79aa2ed3dab55565a7d0be7ab9c489caaf30aac6ebcbc56bb229c61d00d
Giải mã: {'num_beams': 5, 'max_new_tokens': 225, 'silence_peak': 0.001, 'max_tokens_per_second': 12, 'max_tokens_offset': 20}
WER DEV (%): {'B0V': 24.83, 'V3': 18.26, 'V3B': 17.44, 'V4': 17.96, 'V4B': 17.26}
Bộ nhớ commit trống: 6.1 GB 


## 1. Hệ thống cuối V4B


In [2]:
decode_once('V4B_phowhisper_small_lora_vsplit_specaug_beam5', 'test', 'test_beam5')
decode_once('V4B_phowhisper_small_lora_vsplit_specaug_beam5', 'hard', 'hard_beam5')


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V4B_phowhisper_small_lora_vsplit_specaug_beam5 --split test --allow-test --run-id test_beam5 --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261027_014226_171630.log
[INIT] mode=eval experiment=V4B_phowhisper_small_lora_vsplit_specaug_beam5; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v

## 2. Mốc so sánh: B0V (zero-shot), V3, V4 (greedy)


In [3]:
decode_once('B0V_phowhisper_small_zeroshot_vsplit', 'test', 'test_guard')
decode_once('B0V_phowhisper_small_zeroshot_vsplit', 'hard', 'hard_guard')


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval B0V_phowhisper_small_zeroshot_vsplit --split test --allow-test --run-id test_guard --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261027_032151_584617.log
[INIT] mode=eval experiment=B0V_phowhisper_small_zeroshot_vsplit; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "

In [4]:
decode_once('V3_phowhisper_small_lora_vsplit', 'test', 'test_local_v3', '--checkpoint', V3_FINAL)
decode_once('V3_phowhisper_small_lora_vsplit', 'hard', 'hard_local_v3', '--checkpoint', V3_FINAL)


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V3_phowhisper_small_lora_vsplit --split test --allow-test --run-id test_local_v3 --duty-cycle 1.0 --checkpoint "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\runs\V3_phowhisper_small_lora_vsplit\local_v3\final"
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261027_042650_507460.log
[INIT] mode=eval experiment=V3_phowhisper_small_lora_vsplit; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limi

In [5]:
decode_once('V4_phowhisper_small_lora_vsplit_specaug', 'test', 'test_local_v4', '--checkpoint', V4_FINAL)
decode_once('V4_phowhisper_small_lora_vsplit_specaug', 'hard', 'hard_local_v4', '--checkpoint', V4_FINAL)


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V4_phowhisper_small_lora_vsplit_specaug --split test --allow-test --run-id test_local_v4 --duty-cycle 1.0 --checkpoint "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\runs\V4_phowhisper_small_lora_vsplit_specaug\local_v4\final"
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261027_052803_499162.log
[INIT] mode=eval experiment=V4_phowhisper_small_lora_vsplit_specaug; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 6

## 3. (Tùy chọn) V3B — đủ bảng 2×2 SpecAugment × beam


In [7]:
decode_once('V3B_phowhisper_small_lora_vsplit_beam5', 'test', 'test_beam5')
decode_once('V3B_phowhisper_small_lora_vsplit_beam5', 'hard', 'hard_beam5')


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V3B_phowhisper_small_lora_vsplit_beam5 --split test --allow-test --run-id test_beam5 --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261027_073157_642711.log
[INIT] mode=eval experiment=V3B_phowhisper_small_lora_vsplit_beam5; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,

## Kết quả TEST / HARD


In [8]:
import pandas as pd
SYSTEMS = {'B0V': ('B0V_phowhisper_small_zeroshot_vsplit', 'test_guard', 'hard_guard'),
           'V3': ('V3_phowhisper_small_lora_vsplit', 'test_local_v3', 'hard_local_v3'),
           'V3B': ('V3B_phowhisper_small_lora_vsplit_beam5', 'test_beam5', 'hard_beam5'),
           'V4': ('V4_phowhisper_small_lora_vsplit_specaug', 'test_local_v4', 'hard_local_v4'),
           'V4B (cuối)': ('V4B_phowhisper_small_lora_vsplit_specaug_beam5', 'test_beam5', 'hard_beam5')}
rows = {}
for name, (exp, test_run, hard_run) in SYSTEMS.items():
    test, hard = RUNS/exp/test_run/'eval/test/metrics.json', RUNS/exp/hard_run/'eval/hard/metrics.json'
    if not test.is_file():
        continue
    m = json.loads(test.read_text(encoding='utf-8'))
    row = {'TEST overall': 100*m['overall']['wer'], **{f'TEST {s}': 100*v['wer'] for s, v in m['by_source'].items()},
           'TEST term recall': 100*m['overall'].get('cs_term_recall', float('nan'))}
    if hard.is_file():
        h = json.loads(hard.read_text(encoding='utf-8'))
        row |= {'HARD WER': 100*h['overall']['wer'], 'HARD term recall': 100*h['overall'].get('cs_term_recall', float('nan'))}
    rows[name] = row
display(pd.DataFrame(rows).round(2))


,B0V,V3,V3B,V4,V4B (cuối)
TEST overall,23.21,18.46,17.69,17.88,17.38
TEST FLEURS,12.20,9.19,8.83,9.18,8.86
TEST VIVOS,6.28,7.15,6.79,6.66,6.57
TEST ViMedCSS_vsplit,30.02,18.59,18.15,17.97,17.77
TEST VietMed,25.20,22.65,21.54,21.89,21.14
TEST term recall,13.66,55.31,59.40,59.16,62.63
HARD WER,39.68,21.90,21.65,20.00,23.42
HARD term recall,9.52,40.48,42.86,41.67,46.43
